# GVH Diagonal Cubic 0.3.0.1 — Solar-System PPN Reference Extraction
## Extraction traçable des contraintes publiées avant l’analyse GVH

**Auteur : Charlemagne O Laurince**

Ce notebook transforme des résultats publiés en CSV traçables :

\[
\text{publication}\rightarrow\text{extraction documentée}\rightarrow\text{validation}\rightarrow\text{CSV de référence}.
\]

Il ne réanalyse pas les signaux instrumentaux bruts. Il conserve, pour chaque ligne, la valeur, l’incertitude ou la borne, la convention, le type de vraisemblance, la source et l’emplacement dans la source.

Sources principales : Will (2014), Williams–Turyshev–Boggs (LLR), Berche–Medina (Mercure), Li et al. (SKA-VLBI), Sanghai–Clifton (PPNC). Le document Serret (2018) est enregistré mais exclu des contraintes principales.

In [1]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
np.set_printoptions(precision=12, suppress=True)
print("Python :", sys.version)


Python : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


# 1. Configuration

In [2]:
PROJECT_ROOT = Path(".")
CSV_DIR = PROJECT_ROOT / "data" / "raw" / "ppn" / "extracted"
METADATA_DIR = PROJECT_ROOT / "data" / "raw" / "ppn" / "metadata"
EXPORT_DIR = PROJECT_ROOT / "exports"
for directory in [CSV_DIR, METADATA_DIR, EXPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
EXTRACTION_VERSION = "0.3.0.1"
EXTRACTION_DATE_UTC = datetime.now(timezone.utc).date().isoformat()
print(EXTRACTION_VERSION, EXTRACTION_DATE_UTC)


0.3.0.1 2026-08-07


# 2. Niveaux de données

- `PUBLISHED_MEASUREMENT`
- `PUBLISHED_BOUND`
- `REVIEW_SUMMARY_LIMIT`
- `PUBLISHED_DERIVED_PARAMETER`
- `PEDAGOGICAL_SUMMARY`
- `FUTURE_PROJECTION`


In [3]:
ALLOWED_DATA_LEVELS = {"PUBLISHED_MEASUREMENT","PUBLISHED_BOUND","REVIEW_SUMMARY_LIMIT","PUBLISHED_DERIVED_PARAMETER","PEDAGOGICAL_SUMMARY","FUTURE_PROJECTION"}
ALLOWED_LIKELIHOODS = {"gaussian","absolute_upper_bound","summary_accuracy","not_for_likelihood"}


# 3. Registre des sources

In [4]:
sources_df = pd.DataFrame([
{"source_id":"WILL2014_LRR","file_name":"lrr-2014-4(1).pdf","title":"The Confrontation between General Relativity and Experiment","authors":"Clifford M. Will","year":2014,"publication":"Living Reviews in Relativity 17, 4","doi":"10.12942/lrr-2014-4","peer_review_status":"PEER_REVIEWED","source_role":"PEER_REVIEWED_REVIEW","use_in_constraints":True},
{"source_id":"WILLIAMS_LLR","file_name":"0507083.pdf","title":"Lunar Laser Ranging Tests of the Equivalence Principle with the Earth and Moon","authors":"James G. Williams; Slava G. Turyshev; Dale H. Boggs","year":2009,"publication":"International Journal of Modern Physics D / gr-qc/0507083","doi":"","peer_review_status":"PUBLISHED_ARTICLE","source_role":"PRIMARY_CONSTRAINT_SOURCE","use_in_constraints":True},
{"source_id":"BERCHE_MEDINA2024","file_name":"2402.04643.pdf","title":"The advance of Mercury’s perihelion","authors":"Bertrand Berche; Ernesto Medina","year":2024,"publication":"European Journal of Physics submission / arXiv","doi":"","peer_review_status":"PUBLICATION_STATUS_TO_VERIFY","source_role":"SUPPORTING_PEDAGOGICAL_SOURCE","use_in_constraints":False},
{"source_id":"LI_SKA2026","file_name":"2606.25549.pdf","title":"Gravitational Light Deflection with SKA-VLBI and Its Application to Precision Tests of General Relativity","authors":"Y. J. Li et al.","year":2026,"publication":"Advancing Astrophysics with the SKA II","doi":"","peer_review_status":"CHAPTER_OR_PREPRINT","source_role":"FUTURE_FORECAST_SOURCE","use_in_constraints":False},
{"source_id":"SANGHAI_CLIFTON2017","file_name":"1610.08039.pdf","title":"Parameterized Post-Newtonian Cosmology","authors":"Viraj A. A. Sanghai; Timothy Clifton","year":2017,"publication":"Journal article / arXiv:1610.08039","doi":"","peer_review_status":"PUBLISHED_RESEARCH","source_role":"THEORY_CONTEXT","use_in_constraints":False},
{"source_id":"SERRET2018","file_name":"ShapiroSERRETMillenniumjuillet2018.pdf","title":"Shapiro Time Delay derivates from Refraction","authors":"O. Serret","year":2018,"publication":"Millennium Relativity","doi":"","peer_review_status":"NOT_ESTABLISHED","source_role":"EXCLUDED_ALTERNATIVE_SOURCE","use_in_constraints":False}
])
sources_df


,source_id,file_name,title,authors,year,publication,doi,peer_review_status,source_role,use_in_constraints
0,WILL2014_LRR,lrr-2014-4(1).pdf,The Confrontation between General Relativity a...,Clifford M. Will,2014,"Living Reviews in Relativity 17, 4",10.12942/lrr-2014-4,PEER_REVIEWED,PEER_REVIEWED_REVIEW,True
1,WILLIAMS_LLR,0507083.pdf,Lunar Laser Ranging Tests of the Equivalence P...,James G. Williams; Slava G. Turyshev; Dale H. ...,2009,International Journal of Modern Physics D / gr...,,PUBLISHED_ARTICLE,PRIMARY_CONSTRAINT_SOURCE,True
2,BERCHE_MEDINA2024,2402.04643.pdf,The advance of Mercury’s perihelion,Bertrand Berche; Ernesto Medina,2024,European Journal of Physics submission / arXiv,,PUBLICATION_STATUS_TO_VERIFY,SUPPORTING_PEDAGOGICAL_SOURCE,False
3,LI_SKA2026,2606.25549.pdf,Gravitational Light Deflection with SKA-VLBI a...,Y. J. Li et al.,2026,Advancing Astrophysics with the SKA II,,CHAPTER_OR_PREPRINT,FUTURE_FORECAST_SOURCE,False
4,SANGHAI_CLIFTON2017,1610.08039.pdf,Parameterized Post-Newtonian Cosmology,Viraj A. A. Sanghai; Timothy Clifton,2017,Journal article / arXiv:1610.08039,,PUBLISHED_RESEARCH,THEORY_CONTEXT,False
5,SERRET2018,ShapiroSERRETMillenniumjuillet2018.pdf,Shapiro Time Delay derivates from Refraction,O. Serret,2018,Millennium Relativity,,NOT_ESTABLISHED,EXCLUDED_ALTERNATIVE_SOURCE,False


# 4. Contraintes PPN extraites

In [5]:
constraints_df = pd.DataFrame([
{"constraint_id":"PPN_CASSINI_GAMMA_LIMIT","experiment":"Cassini radiometric tracking","observable":"gamma_minus_1","expression":"gamma - 1","central_value":np.nan,"sigma":np.nan,"bound_value":2.3e-5,"bound_side":"absolute","confidence_level":"review_table_limit","likelihood_type":"absolute_upper_bound","unit":"dimensionless","data_level":"REVIEW_SUMMARY_LIMIT","source_id":"WILL2014_LRR","source_page":46,"source_section":"Table 4","source_table":"Table 4","source_equation":"","source_excerpt_summary":"Limit on gamma-1 from Cassini time-delay tracking.","use_in_primary_likelihood":True,"independence_group":"gamma_cassini","notes":"Review reports a bound, not a Gaussian central value."},
{"constraint_id":"PPN_VLBI_GAMMA_2010","experiment":"VLBI global light-deflection analysis","observable":"gamma_minus_1","expression":"gamma - 1","central_value":-0.8e-4,"sigma":1.2e-4,"bound_value":np.nan,"bound_side":"","confidence_level":"1_sigma","likelihood_type":"gaussian","unit":"dimensionless","data_level":"PUBLISHED_MEASUREMENT","source_id":"WILL2014_LRR","source_page":44,"source_section":"4.1.1 The deflection of light","source_table":"","source_equation":"","source_excerpt_summary":"VLBI through 2010: gamma-1=(-0.8±1.2)e-4.","use_in_primary_likelihood":True,"independence_group":"gamma_vlbi_global","notes":""},
{"constraint_id":"PPN_BETA_PERIHELION_LIMIT","experiment":"Mercury perihelion shift with solar J2","observable":"beta_minus_1","expression":"beta - 1","central_value":np.nan,"sigma":np.nan,"bound_value":8.0e-5,"bound_side":"absolute","confidence_level":"review_table_limit","likelihood_type":"absolute_upper_bound","unit":"dimensionless","data_level":"REVIEW_SUMMARY_LIMIT","source_id":"WILL2014_LRR","source_page":46,"source_section":"Table 4 / 4.2","source_table":"Table 4","source_equation":"Eq. 66","source_excerpt_summary":"Review limit on beta-1 from perihelion shift.","use_in_primary_likelihood":True,"independence_group":"beta_mercury","notes":"Uses solar J2=(2.2±0.1)e-7 in the review."},
{"constraint_id":"PPN_LLR_ETA_N","experiment":"Lunar Laser Ranging","observable":"eta_N","expression":"eta_N = 4 beta - gamma - 3","central_value":4.4e-4,"sigma":4.5e-4,"bound_value":np.nan,"bound_side":"","confidence_level":"1_sigma","likelihood_type":"gaussian","unit":"dimensionless","data_level":"PUBLISHED_MEASUREMENT","source_id":"WILLIAMS_LLR","source_page":1,"source_section":"Abstract","source_table":"","source_equation":"eta_N = 4 beta - gamma - 3","source_excerpt_summary":"LLR gives eta_N=(4.4±4.5)e-4.","use_in_primary_likelihood":True,"independence_group":"llr_eta","notes":""},
{"constraint_id":"PPN_LLR_BETA_DERIVED","experiment":"LLR combined with Cassini gamma","observable":"beta_minus_1","expression":"beta - 1","central_value":1.2e-4,"sigma":1.1e-4,"bound_value":np.nan,"bound_side":"","confidence_level":"1_sigma","likelihood_type":"gaussian","unit":"dimensionless","data_level":"PUBLISHED_DERIVED_PARAMETER","source_id":"WILLIAMS_LLR","source_page":1,"source_section":"Abstract","source_table":"","source_equation":"Derived from eta_N and Cassini gamma","source_excerpt_summary":"Published beta-1=(1.2±1.1)e-4.","use_in_primary_likelihood":False,"independence_group":"llr_eta_plus_cassini","notes":"Excluded by default to avoid double counting."},
{"constraint_id":"LLR_EARTH_MOON_EP","experiment":"Lunar Laser Ranging","observable":"delta_MG_over_MI_Earth_Moon","expression":"Delta(M_G/M_I)","central_value":-1.0e-13,"sigma":1.4e-13,"bound_value":np.nan,"bound_side":"","confidence_level":"1_sigma","likelihood_type":"gaussian","unit":"dimensionless","data_level":"PUBLISHED_MEASUREMENT","source_id":"WILLIAMS_LLR","source_page":1,"source_section":"Abstract","source_table":"","source_equation":"","source_excerpt_summary":"Earth-Moon gravitational/inertial mass-ratio inequality.","use_in_primary_likelihood":False,"independence_group":"llr_ep","notes":"Needs a model-specific EP map."},
{"constraint_id":"LLR_SEP_MASS_RATIO","experiment":"LLR plus laboratory WEP tests","observable":"delta_MG_over_MI_SEP","expression":"Delta(M_G/M_I)_SEP","central_value":-2.0e-13,"sigma":2.0e-13,"bound_value":np.nan,"bound_side":"","confidence_level":"1_sigma","likelihood_type":"gaussian","unit":"dimensionless","data_level":"PUBLISHED_DERIVED_PARAMETER","source_id":"WILLIAMS_LLR","source_page":1,"source_section":"Abstract","source_table":"","source_equation":"","source_excerpt_summary":"SEP component from LLR plus laboratory WEP tests.","use_in_primary_likelihood":False,"independence_group":"llr_sep","notes":""}
])
constraints_df


,constraint_id,experiment,observable,expression,central_value,sigma,bound_value,bound_side,confidence_level,likelihood_type,...,data_level,source_id,source_page,source_section,source_table,source_equation,source_excerpt_summary,use_in_primary_likelihood,independence_group,notes
0,PPN_CASSINI_GAMMA_LIMIT,Cassini radiometric tracking,gamma_minus_1,gamma - 1,NaN,NaN,0.000023,absolute,review_table_limit,absolute_upper_bound,...,REVIEW_SUMMARY_LIMIT,WILL2014_LRR,46,Table 4,Table 4,,Limit on gamma-1 from Cassini time-delay track...,True,gamma_cassini,"Review reports a bound, not a Gaussian central..."
1,PPN_VLBI_GAMMA_2010,VLBI global light-deflection analysis,gamma_minus_1,gamma - 1,-8.000000e-05,1.200000e-04,NaN,,1_sigma,gaussian,...,PUBLISHED_MEASUREMENT,WILL2014_LRR,44,4.1.1 The deflection of light,,,VLBI through 2010: gamma-1=(-0.8±1.2)e-4.,True,gamma_vlbi_global,
2,PPN_BETA_PERIHELION_LIMIT,Mercury perihelion shift with solar J2,beta_minus_1,beta - 1,NaN,NaN,0.000080,absolute,review_table_limit,absolute_upper_bound,...,REVIEW_SUMMARY_LIMIT,WILL2014_LRR,46,Table 4 / 4.2,Table 4,Eq. 66,Review limit on beta-1 from perihelion shift.,True,beta_mercury,Uses solar J2=(2.2±0.1)e-7 in the review.
3,PPN_LLR_ETA_N,Lunar Laser Ranging,eta_N,eta_N = 4 beta - gamma - 3,4.400000e-04,4.500000e-04,NaN,,1_sigma,gaussian,...,PUBLISHED_MEASUREMENT,WILLIAMS_LLR,1,Abstract,,eta_N = 4 beta - gamma - 3,LLR gives eta_N=(4.4±4.5)e-4.,True,llr_eta,
4,PPN_LLR_BETA_DERIVED,LLR combined with Cassini gamma,beta_minus_1,beta - 1,1.200000e-04,1.100000e-04,NaN,,1_sigma,gaussian,...,PUBLISHED_DERIVED_PARAMETER,WILLIAMS_LLR,1,Abstract,,Derived from eta_N and Cassini gamma,Published beta-1=(1.2±1.1)e-4.,False,llr_eta_plus_cassini,Excluded by default to avoid double counting.
5,LLR_EARTH_MOON_EP,Lunar Laser Ranging,delta_MG_over_MI_Earth_Moon,Delta(M_G/M_I),-1.000000e-13,1.400000e-13,NaN,,1_sigma,gaussian,...,PUBLISHED_MEASUREMENT,WILLIAMS_LLR,1,Abstract,,,Earth-Moon gravitational/inertial mass-ratio i...,False,llr_ep,Needs a model-specific EP map.
6,LLR_SEP_MASS_RATIO,LLR plus laboratory WEP tests,delta_MG_over_MI_SEP,Delta(M_G/M_I)_SEP,-2.000000e-13,2.000000e-13,NaN,,1_sigma,gaussian,...,PUBLISHED_DERIVED_PARAMETER,WILLIAMS_LLR,1,Abstract,,,SEP component from LLR plus laboratory WEP tests.,False,llr_sep,


# 5. Contexte de Mercure séparé

In [6]:
context_df = pd.DataFrame([
{"context_id":"MERCURY_TOTAL_PRECESSION","quantity":"Observed Mercury perihelion advance","value":5599.745,"unit":"arcsec/century","source_id":"BERCHE_MEDINA2024","source_page":4,"data_level":"PEDAGOGICAL_SUMMARY","use_in_likelihood":False},
{"context_id":"MERCURY_EQUINOX_COMPONENT","quantity":"Equinox-precession contribution","value":5025.645,"unit":"arcsec/century","source_id":"BERCHE_MEDINA2024","source_page":4,"data_level":"PEDAGOGICAL_SUMMARY","use_in_likelihood":False},
{"context_id":"MERCURY_PLANETARY_COMPONENT","quantity":"External-planet contribution","value":531.54,"unit":"arcsec/century","source_id":"BERCHE_MEDINA2024","source_page":1,"data_level":"PEDAGOGICAL_SUMMARY","use_in_likelihood":False},
{"context_id":"MERCURY_RESIDUAL","quantity":"Residual unexplained by Newtonian components","value":42.56,"unit":"arcsec/century","source_id":"BERCHE_MEDINA2024","source_page":1,"data_level":"PEDAGOGICAL_SUMMARY","use_in_likelihood":False}
])
context_df


,context_id,quantity,value,unit,source_id,source_page,data_level,use_in_likelihood
0,MERCURY_TOTAL_PRECESSION,Observed Mercury perihelion advance,5599.745,arcsec/century,BERCHE_MEDINA2024,4,PEDAGOGICAL_SUMMARY,False
1,MERCURY_EQUINOX_COMPONENT,Equinox-precession contribution,5025.645,arcsec/century,BERCHE_MEDINA2024,4,PEDAGOGICAL_SUMMARY,False
2,MERCURY_PLANETARY_COMPONENT,External-planet contribution,531.540,arcsec/century,BERCHE_MEDINA2024,1,PEDAGOGICAL_SUMMARY,False
3,MERCURY_RESIDUAL,Residual unexplained by Newtonian components,42.560,arcsec/century,BERCHE_MEDINA2024,1,PEDAGOGICAL_SUMMARY,False


# 6. Prévisions futures exclues

In [7]:
forecast_df = pd.DataFrame([
{"forecast_id":"SKA_SOLAR_GAMMA","experiment":"Future SKA-VLBI solar light deflection","observable":"gamma_accuracy","forecast_value":1.0e-7,"unit":"dimensionless","source_id":"LI_SKA2026","source_page":1,"data_level":"FUTURE_PROJECTION","use_in_current_likelihood":False},
{"forecast_id":"SKA_JUPITER_GAMMA","experiment":"Future SKA-VLBI Jupiter light deflection","observable":"gamma_accuracy","forecast_value":1.0e-4,"unit":"dimensionless","source_id":"LI_SKA2026","source_page":1,"data_level":"FUTURE_PROJECTION","use_in_current_likelihood":False}
])
forecast_df


,forecast_id,experiment,observable,forecast_value,unit,source_id,source_page,data_level,use_in_current_likelihood
0,SKA_SOLAR_GAMMA,Future SKA-VLBI solar light deflection,gamma_accuracy,1.000000e-07,dimensionless,LI_SKA2026,1,FUTURE_PROJECTION,False
1,SKA_JUPITER_GAMMA,Future SKA-VLBI Jupiter light deflection,gamma_accuracy,1.000000e-04,dimensionless,LI_SKA2026,1,FUTURE_PROJECTION,False


# 7. Correspondance théorie–observables

\[
\gamma_{\rm GR}=1,\quad \beta_{\rm GR}=1,\quad \eta_N=4\beta-\gamma-3=0.
\]


In [8]:
observable_map_df = pd.DataFrame([
{"observable":"gamma_minus_1","model_expression":"gamma(theta)-1","GR_value":0.0},
{"observable":"beta_minus_1","model_expression":"beta(theta)-1","GR_value":0.0},
{"observable":"eta_N","model_expression":"4*beta(theta)-gamma(theta)-3","GR_value":0.0},
{"observable":"perihelion_ppn_factor","model_expression":"(2+2*gamma(theta)-beta(theta))/3","GR_value":1.0}
])
observable_map_df


,observable,model_expression,GR_value
0,gamma_minus_1,gamma(theta)-1,0.0
1,beta_minus_1,beta(theta)-1,0.0
2,eta_N,4*beta(theta)-gamma(theta)-3,0.0
3,perihelion_ppn_factor,(2+2*gamma(theta)-beta(theta))/3,1.0


# 8. Validation et double comptage

In [9]:
def validate_row(row):
    errors=[]
    if row["data_level"] not in ALLOWED_DATA_LEVELS: errors.append("data_level")
    if row["likelihood_type"] not in ALLOWED_LIKELIHOODS: errors.append("likelihood")
    if row["source_id"] not in set(sources_df["source_id"]): errors.append("source")
    if row["likelihood_type"]=="gaussian" and not (np.isfinite(row["central_value"]) and np.isfinite(row["sigma"]) and row["sigma"]>0): errors.append("gaussian")
    if row["likelihood_type"]=="absolute_upper_bound" and not (np.isfinite(row["bound_value"]) and row["bound_value"]>0 and row["bound_side"]=="absolute"): errors.append("bound")
    return errors
constraint_validation_df = pd.DataFrame([
    {"constraint_id":r["constraint_id"],"validation_pass":len(validate_row(r))==0,"errors":";".join(validate_row(r))}
    for _,r in constraints_df.iterrows()
])
dependency_notes_df = pd.DataFrame([
{"constraint_a":"PPN_CASSINI_GAMMA_LIMIT","constraint_b":"PPN_LLR_BETA_DERIVED","dependency":"Derived beta uses Cassini gamma.","default_action":"Do not use both as independent factors."},
{"constraint_a":"PPN_LLR_ETA_N","constraint_b":"PPN_LLR_BETA_DERIVED","dependency":"Derived beta uses eta_N.","default_action":"Use eta_N plus independent gamma, or derived beta alone."}
])
constraint_validation_df


,constraint_id,validation_pass,errors
0,PPN_CASSINI_GAMMA_LIMIT,True,
1,PPN_VLBI_GAMMA_2010,True,
2,PPN_BETA_PERIHELION_LIMIT,True,
3,PPN_LLR_ETA_N,True,
4,PPN_LLR_BETA_DERIVED,True,
5,LLR_EARTH_MOON_EP,True,
6,LLR_SEP_MASS_RATIO,True,


# 9. Catalogue primaire

In [10]:
primary_constraints_df = constraints_df[constraints_df["use_in_primary_likelihood"].astype(bool)].copy()
primary_constraints_df[["constraint_id","observable","central_value","sigma","bound_value","likelihood_type","source_id"]]


,constraint_id,observable,central_value,sigma,bound_value,likelihood_type,source_id
0,PPN_CASSINI_GAMMA_LIMIT,gamma_minus_1,NaN,NaN,0.000023,absolute_upper_bound,WILL2014_LRR
1,PPN_VLBI_GAMMA_2010,gamma_minus_1,-0.00008,0.00012,NaN,gaussian,WILL2014_LRR
2,PPN_BETA_PERIHELION_LIMIT,beta_minus_1,NaN,NaN,0.000080,absolute_upper_bound,WILL2014_LRR
3,PPN_LLR_ETA_N,eta_N,0.00044,0.00045,NaN,gaussian,WILLIAMS_LLR


# 10. Contrôle GR

In [11]:
GR = {"gamma_minus_1":0.0,"beta_minus_1":0.0,"eta_N":0.0}
def evaluate(row, value):
    if row["likelihood_type"]=="gaussian":
        return ((value-row["central_value"])/row["sigma"])**2, True
    if row["likelihood_type"]=="absolute_upper_bound":
        return np.nan, abs(value)<=row["bound_value"]
    return np.nan, np.nan
gr_rows=[]
for _,row in primary_constraints_df.iterrows():
    if row["observable"] in GR:
        chi2, passed = evaluate(row, GR[row["observable"]])
        gr_rows.append({"constraint_id":row["constraint_id"],"observable":row["observable"],"GR_prediction":GR[row["observable"]],"chi2":chi2,"pass_bound":passed})
gr_control_df = pd.DataFrame(gr_rows)
gr_control_df


,constraint_id,observable,GR_prediction,chi2,pass_bound
0,PPN_CASSINI_GAMMA_LIMIT,gamma_minus_1,0.0,NaN,True
1,PPN_VLBI_GAMMA_2010,gamma_minus_1,0.0,0.444444,True
2,PPN_BETA_PERIHELION_LIMIT,beta_minus_1,0.0,NaN,True
3,PPN_LLR_ETA_N,eta_N,0.0,0.956049,True


# 11. Validation globale

In [12]:
validation_df = pd.DataFrame([
{"test":"unique constraints","status":"PASS" if constraints_df["constraint_id"].is_unique else "FAIL"},
{"test":"unique sources","status":"PASS" if sources_df["source_id"].is_unique else "FAIL"},
{"test":"valid rows","status":"PASS" if constraint_validation_df["validation_pass"].all() else "FAIL"},
{"test":"excluded Serret source not used","status":"PASS" if not constraints_df["source_id"].eq("SERRET2018").any() else "FAIL"},
{"test":"future forecasts excluded","status":"PASS" if not forecast_df["use_in_current_likelihood"].any() else "FAIL"},
{"test":"derived beta not primary","status":"PASS" if not constraints_df.loc[constraints_df["constraint_id"]=="PPN_LLR_BETA_DERIVED","use_in_primary_likelihood"].iloc[0] else "FAIL"},
{"test":"GR satisfies hard bounds","status":"PASS" if gr_control_df["pass_bound"].dropna().all() else "FAIL"}
])
overall_status = "PASS-SOLAR-SYSTEM-REFERENCE-EXTRACTION" if validation_df["status"].eq("PASS").all() else "CHECK"
print(overall_status)
validation_df


PASS-SOLAR-SYSTEM-REFERENCE-EXTRACTION


,test,status
0,unique constraints,PASS
1,unique sources,PASS
2,valid rows,PASS
3,excluded Serret source not used,PASS
4,future forecasts excluded,PASS
5,derived beta not primary,PASS
6,GR satisfies hard bounds,PASS


# 12. Export

In [13]:
sources_df.to_csv(CSV_DIR/"solar_system_sources_catalog.csv",index=False)
constraints_df.to_csv(CSV_DIR/"solar_system_ppn_constraints_full.csv",index=False)
primary_constraints_df.to_csv(CSV_DIR/"solar_system_ppn_constraints_primary.csv",index=False)
context_df.to_csv(CSV_DIR/"mercury_historical_context.csv",index=False)
forecast_df.to_csv(CSV_DIR/"solar_system_future_forecasts.csv",index=False)
observable_map_df.to_csv(CSV_DIR/"ppn_observable_model_map.csv",index=False)
dependency_notes_df.to_csv(CSV_DIR/"ppn_constraint_dependencies.csv",index=False)
constraint_validation_df.to_csv(EXPORT_DIR/"GVH_Diagonal_Cubic_0.3.0.1_Constraint_Validation.csv",index=False)
gr_control_df.to_csv(EXPORT_DIR/"GVH_Diagonal_Cubic_0.3.0.1_GR_Control.csv",index=False)
validation_df.to_csv(EXPORT_DIR/"GVH_Diagonal_Cubic_0.3.0.1_Validation.csv",index=False)
metadata={"notebook":"GVH_Diagonal_Cubic_0.3.0.1_Solar_System_PPN_Reference_Extraction","status":overall_status,"extraction_version":EXTRACTION_VERSION,"extraction_date_utc":EXTRACTION_DATE_UTC,"source_count":len(sources_df),"constraint_count":len(constraints_df),"primary_constraint_count":len(primary_constraints_df),"raw_instrument_data_used":False,"published_summary_constraints_used":True,"excluded_source_ids":["SERRET2018"],"future_forecasts_used_in_likelihood":False}
with (METADATA_DIR/"solar_system_ppn_extraction_metadata.json").open("w",encoding="utf-8") as f: json.dump(metadata,f,indent=2,ensure_ascii=False)
print("Exports terminés dans", CSV_DIR)


Exports terminés dans data/raw/ppn/extracted


# 13. Conclusion

Le notebook crée :

- `solar_system_ppn_constraints_full.csv` : toutes les contraintes et valeurs dérivées ;
- `solar_system_ppn_constraints_primary.csv` : catalogue par défaut pour 0.3.4 ;
- un registre de sources ;
- un contexte historique de Mercure ;
- un registre des dépendances empêchant le double comptage.

Il ne prétend pas reproduire les signaux Cassini, les séries temporelles LLR, les observations VLBI individuelles ni les ajustements d’éphémérides. Il formalise les contraintes numériques publiées et leur provenance.